# OpenMM 2000-Step GPCRmd Preview

This notebook previews the local OpenMM artifact generated from the selected GPCRmd prepared system. The default artifact is a 2000-step OpenMM `OpenCL` restrained preview sampled every 20 steps, giving 101 saved frames.

The ligand restraint target is translated 8 A along x during the run so the notebook visibly previews ligand translational motion. This is a visualization smoke run, not force-field parity: PME, constraints, full bonded terms, and production GPCRmd protocol parity are intentionally out of scope for this preview artifact.

In [ ]:
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "helpers").exists():
    NOTEBOOK_DIR = Path("notebooks/ligand-receptor-motion").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers.motion_analysis import (  # noqa: E402
    ProcessedTrajectory,
    analysis_tables,
    ligand_com_displacement,
    trajectory_quality_report,
)
from helpers.visualization import make_ligand_motion_figure, playback_table  # noqa: E402

DATASET_NAME = os.environ.get("OPENMM_PREVIEW_DATASET", "729-2000-opencl-restrained-preview")
DATA_DIR = NOTEBOOK_DIR / "data/openmm-preview" / DATASET_NAME
TRAJECTORY_PATH = DATA_DIR / "processed_trajectory.npz"
REPORT_PATH = DATA_DIR / "openmm_preview_run_report.json"
VIEW_STRIDE = int(os.environ.get("OPENMM_PREVIEW_VIEW_STRIDE", "1"))

## Artifact Status

Regenerate the default guided preview outside the notebook with:

```bash
uv run python scripts/run_openmm_gpcrmd_preview.py \
  --out notebooks/ligand-receptor-motion/data/openmm-preview/729-2000-opencl-restrained-preview \
  --steps 2000 --sample-interval 20 --platform OpenCL \
  --dt-ps 0.0005 --temperature 300 --friction 1 \
  --restraint-k 1000 --ligand-restraint-k 50000 \
  --ligand-translation-A 8 0 0 --nonbonded-mode none
```

In [ ]:
if not TRAJECTORY_PATH.exists() or not REPORT_PATH.exists():
    missing = [str(path) for path in (TRAJECTORY_PATH, REPORT_PATH) if not path.exists()]
    raise FileNotFoundError(f"Missing OpenMM preview artifact: {missing}")

trajectory = ProcessedTrajectory.load(TRAJECTORY_PATH)
report = json.loads(REPORT_PATH.read_text())
source = trajectory.source

display(
    pd.DataFrame(
        [
            {
                "dataset": DATASET_NAME,
                "status": report.get("status"),
                "engine": report.get("engine"),
                "workflow": report.get("workflow"),
                "platform": report.get("platform"),
                "device": report.get("platform_properties", {}).get("DeviceName"),
                "steps": report.get("steps"),
                "sample_interval": report.get("sample_interval"),
                "saved_frames": report.get("sampled_frame_count"),
                "simulated_time_ps": report.get("simulated_time_ps"),
                "dt_ps": report.get("dt_ps"),
                "steps_per_second": report.get("integration_steps_per_second"),
                "nonbonded_mode": source.get("nonbonded_mode"),
                "restraint_k": source.get("restraint_k_kj_mol_nm2"),
                "ligand_restraint_k": source.get("ligand_restraint_k_kj_mol_nm2"),
                "ligand_translation_A": source.get("ligand_translation_A"),
            }
        ]
    )
)

display(Markdown(f"**Preview note:** {source.get('note', '')}"))
display(playback_table(trajectory))

quality_report = trajectory_quality_report(trajectory, max_constraint_error_A=0.0)
display(Markdown("**Visualization Quality Report**"))
display(pd.DataFrame([{k: v for k, v in quality_report.items() if k != "warnings"}]))
for warning in quality_report["warnings"]:
    display(Markdown(f"- {warning}"))

## Trajectory Viewer

The viewer shows saved OpenMM preview frames in a PBC-corrected display frame. With the default settings, it displays 101 frames from 2000 integration steps and the ligand translates about 8 A along the guided target path.

In [ ]:
if VIEW_STRIDE > 1:
    viewer_trajectory = replace(
        trajectory,
        positions=trajectory.positions[::VIEW_STRIDE],
        time_ps=trajectory.time_ps[::VIEW_STRIDE],
        source={**trajectory.source, "viewer_stride": VIEW_STRIDE},
    )
else:
    viewer_trajectory = trajectory

figure = make_ligand_motion_figure(
    viewer_trajectory,
    title=(
        "OpenMM GPCRmd 2000-step restrained preview "
        f"({viewer_trajectory.frame_count} displayed frames)"
    ),
    frame_interval_ms=60,
)
figure.show()

## Diagnostics

In [ ]:
diagnostics = pd.DataFrame(
    {
        "step": report["sample_interval"] * pd.Series(range(report["sampled_frame_count"])),
        "time_ps": trajectory.time_ps,
        "potential_energy_kj_mol": report["potential_energy_kj_mol"],
        "kinetic_energy_kj_mol": report["kinetic_energy_kj_mol"],
        "temperature_K": report["temperature_K"],
        "ligand_com_displacement_A": ligand_com_displacement(trajectory),
    }
)
display(diagnostics.tail())

energy_fig = px.line(
    diagnostics,
    x="time_ps",
    y=["potential_energy_kj_mol", "kinetic_energy_kj_mol"],
    title="OpenMM preview energy diagnostics",
    labels={"value": "kJ/mol", "variable": "term"},
)
energy_fig.show()

temp_fig = px.line(
    diagnostics,
    x="time_ps",
    y="temperature_K",
    title="OpenMM preview temperature",
)
temp_fig.show()

motion_fig = px.line(
    diagnostics,
    x="time_ps",
    y="ligand_com_displacement_A",
    title="Ligand COM displacement in saved OpenMM preview frames",
)
motion_fig.show()

## Interaction Tables

In [ ]:
tables = analysis_tables(trajectory)
for name, table in tables.items():
    display(Markdown(f"**{name.replace('_', ' ').title()}**"))
    display(table)